# YouTube comments — data preprocessing

In [13]:
# Install the libraries used by this preprocessing notebook
%pip install -q numpy pandas scikit-learn nltk

# Download the WordNet dictionaries used by NLTK lemmatization
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [14]:
# Import tools for file paths, text normalization, and data processing
from pathlib import Path
import html
import re
import unicodedata

import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

# Define the location of the raw comments dataset
DATA_PATH = Path("data/youtube_comments.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/youtube_comments.csv')
# Load only the columns needed for sentiment preprocessing
REQUIRED_COLUMNS = ['VideoTitle', 'CommentText']
# Initialize the NLTK WordNet lemmatizer
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
lemmatizer = WordNetLemmatizer()
print('NLTK WordNet lemmatizer loaded successfully!')

NLTK WordNet lemmatizer loaded successfully!


In [15]:
df = pd.read_csv(DATA_PATH, usecols=REQUIRED_COLUMNS)
df.head()

,VideoTitle,CommentText
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”


## Inspect the raw dataset

In [16]:
# Display the number of rows and columns
print('Shape:', df.shape)
# Check for missing values in each column
print('Columns:', df.columns.tolist())
display(df.isna().sum().to_frame('missing'))
# Count completely duplicated rows before preprocessing
print('Duplicate rows:', int(df.duplicated().sum()))
# Calculate basic comment-length statistics
df['raw_comment_length'] = df['CommentText'].fillna('').astype(str).str.len()
df['raw_token_count'] = df['CommentText'].fillna('').astype(str).str.split().str.len()
display(df[['raw_comment_length', 'raw_token_count']].describe().round(2))

Shape: (924255, 2)
Columns: ['VideoTitle', 'CommentText']


,missing
VideoTitle,0
CommentText,383


Duplicate rows: 175376


,raw_comment_length,raw_token_count
count,924255.00,924255.00
mean,56.41,10.77
std,133.75,23.59
min,0.00,0.00
25%,15.00,3.00
50%,33.00,6.00
75%,63.00,12.00
max,13780.00,2347.00


## Remove non-English comments

In [17]:
# Identify comments that are mainly written in English
def is_english_comment(text: str) -> bool:
    # Convert HTML entities and other values to text
    text = html.unescape(str(text))
    # Keep alphabetic characters for language detection
    letters = [char for char in text if char.isalpha()]
    if len(letters) < 2:
        return False
    # Calculate how many letters belong to the Latin alphabet
    latin_ratio = sum(char.lower() in 'abcdefghijklmnopqrstuvwxyz' for char in letters) / len(letters)
    if latin_ratio < 0.85:
        return False
    words = re.findall(r'[A-Za-z]+', text.lower())
    common_english_words = sum(word in ENGLISH_STOP_WORDS for word in words)
    return latin_ratio >= 0.95 or common_english_words > 0

# Apply the English-language check to every comment
df['is_english'] = df['CommentText'].fillna('').map(is_english_comment)
# Keep only comments identified as English
print(df['is_english'].value_counts())
df = df[df['is_english']].copy()
df = df.drop(columns='is_english')
print('Rows after English-language filtering:', len(df))

is_english
True     900590
False     23665
Name: count, dtype: int64
Rows after English-language filtering: 900590


## Clean, tokenize, and lemmatize with NLTK WordNet

In [18]:
# Define the text-cleaning patterns
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"\b[a-z]+\b", re.IGNORECASE)

# Keep words that can change sentiment or meaning
PROTECTED_WORDS = {
    'no', 'not', 'never', 'neither', 'none', 'nobody',
    'nothing', 'nowhere', 'cannot', 'but', 'however',
    'although', 'though', 'whereas', 'yet', 'very', 'too',
    'most', 'more', 'less', 'rather', 'enough'
}
CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS) - PROTECTED_WORDS

# Expand contractions before punctuation removal so negation is preserved
def prepare_text(text: str) -> str:
    text = html.unescape(str(text))
    text = unicodedata.normalize('NFKC', text).lower()
    text = URL_RE.sub(' ', text)
    text = re.sub(r"\bcan't\b", 'can not', text)
    text = re.sub(r"\bwon't\b", 'will not', text)
    text = re.sub(r"n't\b", ' not', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return WHITESPACE_RE.sub(' ', text).strip()

# Clean, remove non-sentiment stopwords, and lemmatize with WordNet
def clean_and_tokenize(text: str) -> tuple[str, list[str]]:
    prepared = prepare_text(text)
    raw_tokens = WORD_RE.findall(prepared)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in raw_tokens
        if token not in CUSTOM_STOPWORDS and len(token) > 1
    ]
    return ' '.join(tokens), tokens

# Apply the same preprocessing to every comment
cleaned = df['CommentText'].fillna('').map(clean_and_tokenize)
df['clean_comment'] = cleaned.str[0]
df['comment_tokens'] = cleaned.str[1]

# Remove comments that became empty after cleaning
df = df[df['clean_comment'].str.len().gt(0)].copy()
df[['CommentText', 'clean_comment', 'comment_tokens']].head(10)

,CommentText,clean_comment,comment_tokens
0,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim youtube short,"[claim, youtube, short]"
1,I will try these too 😊,try too,"[try, too]"
2,Career ladder,career ladder,"[career, ladder]"
3,Very helpful,very helpful,"[very, helpful]"
4,Im not proud of doing “brainrot”,im not proud doing brainrot,"[im, not, proud, doing, brainrot]"
5,Thank you bro for info❤,thank bro info,"[thank, bro, info]"
6,What about financial documentary,financial documentary,"[financial, documentary]"
7,oh for gods sake get to the point. You are tal...,oh god sake point talking issue instead just s...,"[oh, god, sake, point, talking, issue, instead..."
8,If we upload other’s clips in our videos dont ...,upload clip video dont ask credit famous ppl,"[upload, clip, video, dont, ask, credit, famou..."
10,great vudei,great vudei,"[great, vudei]"


## Test negation-aware preprocessing

In [19]:
# Test positive, negative, and neutral comments with different negation forms
test_comments = [
    'This is an amazing video!',
    'This is not good at all.',
    'I never liked the editing.',
    "I don't hate this video.",
    'What camera did you use?'
]

# Run the same NLTK preprocessing and display the generated features
for text in test_comments:
    cleaned_text, features = clean_and_tokenize(text)
    print(f'{text} -> {cleaned_text}')

This is an amazing video! -> amazing video
This is not good at all. -> not good
I never liked the editing. -> never liked editing
I don't hate this video. -> not hate video
What camera did you use? -> camera did use


## Remove duplicate comments after tokenization

In [20]:
# Count rows before removing normalized duplicates
before_duplicates = len(df)

# Remove comments from the same video with identical cleaned text
df = df.drop_duplicates(
    subset=['VideoTitle', 'clean_comment']
).reset_index(drop=True)

# Report how many duplicate comments were removed
print(f'Removed duplicate comments: {before_duplicates - len(df)}')
print(f'Rows remaining: {len(df)}')

Removed duplicate comments: 262872
Rows remaining: 630935


## Convert comments to TF-IDF features

In [21]:
# Configure TF-IDF to create unigram and bigram features
tfidf = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
# Learn the vocabulary and convert comments into numerical vectors
X_tfidf = tfidf.fit_transform(df['clean_comment'])
print('Rows:', X_tfidf.shape[0])
print('Features:', X_tfidf.shape[1])

Rows: 630935
Features: 415811


## Preprocessing output

In [22]:
# Preview the original and cleaned comment columns
df[['VideoTitle', 'CommentText', 'clean_comment', 'comment_tokens']].head()
# Display the TF-IDF matrix dimensions
X_tfidf.shape

(630935, 415811)

## Save cleaned data

In [23]:
# Remove temporary analysis columns before exporting
clean_data = df.drop(columns=['raw_comment_length', 'raw_token_count', 'comment_tokens'], errors='ignore').copy()
# Place the cleaned text column at the end of the output
clean_data = clean_data[[column for column in clean_data.columns if column != 'clean_comment'] + ['clean_comment']]
# Add empty label columns so the Gemini pipeline can fill them later
clean_data['Sentiment'] = ''
clean_data['Sentiment_label'] = ''
# Define the output file path
OUTPUT_PATH = DATA_PATH.with_name('youtube_comments_cleaned.csv')
# Save the cleaned dataset without the pandas index
clean_data.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Saved: {OUTPUT_PATH}')
clean_data.head()

Saved: data/youtube_comments_cleaned.csv


,VideoTitle,CommentText,clean_comment,Sentiment,Sentiment_label
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim youtube short,,
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊,try too,,
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder,career ladder,,
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful,very helpful,,
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”,im not proud doing brainrot,,


In [25]:
# Confirm the selected preprocessing implementation
print('Preprocessing implementation: NLTK WordNet')
clean_data.head(10)

Preprocessing implementation: NLTK WordNet


,VideoTitle,CommentText,clean_comment,Sentiment,Sentiment_label
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim youtube short,,
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊,try too,,
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder,career ladder,,
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful,very helpful,,
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”,im not proud doing brainrot,,
5,The ONLY 8 YouTube Shorts Niches That Will Mak...,Thank you bro for info❤,thank bro info,,
6,The ONLY 8 YouTube Shorts Niches That Will Mak...,What about financial documentary,financial documentary,,
7,The ONLY 8 YouTube Shorts Niches That Will Mak...,oh for gods sake get to the point. You are tal...,oh god sake point talking issue instead just s...,,
8,The ONLY 8 YouTube Shorts Niches That Will Mak...,If we upload other’s clips in our videos dont ...,upload clip video dont ask credit famous ppl,,
9,The ONLY 8 YouTube Shorts Niches That Will Mak...,great vudei,great vudei,,
